# Sprint 1 — Conhecendo os dados
## Dengue em Santa Catarina: coleta, integração e análise exploratória

Este notebook baixa todas as fontes públicas usadas no projeto, monta a **base analítica mensal por
município** (`data/processed/base_analitica_sc.csv`) e explora as hipóteses que orientam o modelo de
risco (H1–H7).

**Como executar**
- **Localmente**, na raiz do repositório: `uv sync` e abra este notebook com o kernel do projeto.
- **No Google Colab**, faça o upload do notebook e escolha *Ambiente de execução > Executar tudo*. A
  primeira célula clona o repositório e instala as dependências.

Os downloads ficam em `data/raw/` e são reaproveitados nas execuções seguintes. As flags da seção 0
forçam um novo download quando houver dados mais recentes. O mesmo código está nos módulos de
`src/` e no script `scripts/update_base.py`.

### Onde este notebook se encaixa

| Etapa | Material | Resultado |
|---|---|---|
| **1. Dados e análise exploratória** | **este notebook** | Base analítica e conclusões sobre as hipóteses H1–H7 |
| 2. Modelagem | [`notebook_modelo_baseline.ipynb`](notebook_modelo_baseline.ipynb) | Modelo de risco, métricas e base do painel |
| 3. Painel | [https://dengue-sc.streamlit.app/](https://dengue-sc.streamlit.app/) | Resultados em linguagem simples para o público |
| 4. Artigo | em elaboração | Pesquisa, métodos e discussão |

Definições e métodos resumidos estão em [`docs/modelo-e-validacao.md`](../docs/modelo-e-validacao.md).

## 0. Configuração

In [ ]:
# Ambiente: no Google Colab, clona o repositório e instala as dependências; localmente, usa o projeto.
import sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    !git clone -q https://github.com/EmersonComar/planejamento_gestao_projetos.git
    %cd planejamento_gestao_projetos
    !pip install -q "scikit-learn==1.6.1" "pandas>=2.2" duckdb pyarrow pypdf plotly joblib

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "app").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
print(f"Raiz do projeto: {ROOT} | Colab: {EM_COLAB}")

In [ ]:
from src.application.use_cases import base_analitica
from src.application.use_cases.build_analytics_data import centroids_from_geojson
from src.infrastructure import pipeline
from src.infrastructure.sources import dive, ibge, inmet

paths = pipeline.Paths(ROOT)
ARQ_BASE_ANALITICA = paths.base

# Baixar de novo mesmo se já houver cópia local (use True para trazer dados novos)
ATUALIZAR_INFODENGUE = False        # série semanal completa (~10 min)
ATUALIZAR_INMET_ANO_CORRENTE = False  # o zip do ano corrente é parcial e cresce a cada mês
ATUALIZAR_INFORMES_DIVE = False

ULTIMO_MES = pipeline.last_complete_month()
print(f"Período: {pipeline.FIRST_YEAR}-01 a {ULTIMO_MES} (último mês completo)")

## 1. Fontes e definições

| Fonte | O que fornece | Granularidade |
|---|---|---|
| [InfoDengue (Fiocruz)](https://info.dengue.mat.br/) — API `alertcity` | Série municipal de casos prováveis (`casprov`), notificações (`casos`), temperatura e umidade; usada nas análises municipais e no modelo | Município × semana epidemiológica |
| [IBGE — Localidades e Malhas](https://servicodados.ibge.gov.br/api/docs/) | Municípios, regiões e mapa | Município |
| [IBGE — SIDRA](https://sidra.ibge.gov.br/) tabelas 6579 e 4714 | População (estimativas anuais e Censo 2022) e área (Censo 2022) | Município × ano |
| [INMET — Dados Históricos](https://portal.inmet.gov.br/dadoshistoricos) | Precipitação horária das estações automáticas | Estação × hora |
| [ANA — Atlas Esgotos (ETEs 2020)](https://dadosabertos.ana.gov.br/) | Estações de tratamento de esgoto e remoção de DBO | Município |
| [DIVE/SC — Informes epidemiológicos](https://dive.saude.sc.gov.br/index.php/doencas-e-agravos/dengue) | Totais estaduais oficiais por mês de início dos sintomas; mantidos como referência separada, não usados no modelo | Estado × mês |

**Definições usadas em todo o projeto**
- **Caso**: *caso provável* de dengue — confirmados, suspeitos e inconclusivos, sem os descartados.
  É o conceito da Diretoria de Vigilância Epidemiológica de SC. As notificações totais, que incluem
  os descartados, ficam numa coluna à parte.
- **Incidência**: casos prováveis do mês ÷ população do ano × 100 mil habitantes.
- **Mês epidêmico**: incidência mensal ≥ 300 casos prováveis por 100 mil habitantes.

## 2. Coleta
### 2.1 Municípios, regiões e malha (IBGE)

In [ ]:
municipios, malha = pipeline.collect_geography(paths)
print(f"{len(municipios)} municípios; malha com {len(malha['features'])} polígonos")
municipios.head()

### 2.2 Casos e clima (InfoDengue)

Uma consulta por município traz toda a série semanal de 2015 até o ano corrente, incluindo a semana
epidemiológica 53 dos anos que a têm. Campos usados:
- `casprov`: casos prováveis;
- `casos`: notificações;
- `tempmed`, `tempmin`, `tempmax`: temperatura;
- `umidmed`, `umidmin`, `umidmax`: umidade.

In [ ]:
semanal = pipeline.collect_infodengue(paths, refresh=ATUALIZAR_INFODENGUE)
semanal["data_semana"] = pd.to_datetime(semanal["data_iniSE"], unit="ms")
print(f"{len(semanal):,} semanas de {semanal['codigo_ibge'].nunique()} municípios, "
      f"{semanal['data_semana'].min():%d/%m/%Y} a {semanal['data_semana'].max():%d/%m/%Y}")
semanal[["codigo_ibge", "nome_municipio", "data_semana", "SE", "casprov", "casos", "tempmed", "umidmed"]].tail()

### 2.3 População e área (IBGE)

- **População**: estimativas anuais da tabela 6579 e, em 2022, o Censo (tabela 4714). Em 2023 o IBGE
  não publicou estimativa municipal; o valor é interpolado entre 2022 e 2024.
- **Área**: Censo 2022, que inclui os municípios criados depois de 2010 (Pescaria Brava e Balneário
  Rincão).

In [ ]:
populacao, area = pipeline.collect_ibge_tables(paths)
resumo_pop = populacao.groupby("ano").agg(populacao_sc=("populacao", "sum"), origem=("origem", "first"))
print(f"Área de {len(area)} municípios (Censo 2022)")
resumo_pop

### 2.4 Chuva (INMET)

O INMET publica um zip por ano com os dados horários de todas as estações automáticas do país, com
cerca de 100 MB cada. Somamos a precipitação horária por mês e registramos quantas horas tiveram
medição. Um mês com muitas horas sem medição teria a chuva subestimada se as lacunas fossem tratadas
como zero.

In [ ]:
pipeline.collect_inmet(paths, refresh_current_year=ATUALIZAR_INMET_ANO_CORRENTE, progress=None)
chuva_estacoes, estacoes = pipeline.process_inmet(paths)
chuva_estacoes["cobertura"] = chuva_estacoes["horas_medidas"] / (chuva_estacoes["ano_mes"].dt.days_in_month * 24)
print(f"{len(estacoes)} estações automáticas em SC; {len(chuva_estacoes):,} meses de estação")
print(f"Meses sem nenhuma hora medida: {(chuva_estacoes['horas_medidas'] == 0).mean():.1%}; "
      f"com menos de 70% das horas: {(chuva_estacoes['cobertura'] < 0.7).mean():.1%}")
chuva_estacoes.groupby(chuva_estacoes["ano_mes"].dt.year)["cobertura"].apply(lambda c: (c >= 0.7).mean()).rename(
    "meses com ≥70% das horas medidas").to_frame().T.round(2)

**Regra adotada:**
- Um mês de estação só é usado quando teve ao menos **70% das horas medidas**.
- Cada município usa a **estação mais próxima do seu centro que mediu aquele mês**, até 60 km.
- Se nenhuma estação a até 60 km mediu o mês, a chuva fica ausente. Nada é preenchido com zero.

In [ ]:
associacao = pipeline.map_stations(paths)
primaria = associacao.loc[associacao["ordem"] == 1]
print(f"Municípios com estação a até 60 km: {associacao['codigo_ibge'].nunique()} de {len(municipios)}; "
      f"distância mediana até a estação mais próxima: {primaria['distancia_km'].median():.0f} km")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(primaria["distancia_km"], bins=20, color="#1c7ed6")
ax.set(xlabel="Distância até a estação mais próxima (km)", ylabel="Municípios")
plt.show()

### 2.5 Saneamento (ANA)

In [ ]:
saneamento = pipeline.collect_sanitation(paths)
print(f"{len(saneamento)} de {len(municipios)} municípios têm ETE ativa no Atlas Esgotos")
saneamento.describe().round(1)

### 2.6 Referência oficial (DIVE/SC)

A DIVE/SC é a fonte oficial dos totais estaduais publicados. Esses informes são mantidos como referência
separada da série municipal do InfoDengue e não alimentam a base do modelo.

A tabela apresenta casos notificados por mês de início dos sintomas e classificação final:
- dengue;
- dengue com sinais de alarme;
- dengue grave;
- descartado;
- inconclusivo;
- suspeito.

Os casos prováveis são a soma de todas as classes exceto "descartado". A tabela é extraída
automaticamente do PDF de cada informe.

In [ ]:
referencia = pipeline.collect_dive_reference(paths, refresh=ATUALIZAR_INFORMES_DIVE)
referencia.groupby(["ano", "documento", "dados_ate"])[["casos_provaveis", "notificacoes"]].sum()

## 3. Integração: base analítica mensal

In [ ]:
chuva_municipios = base_analitica.precipitation_by_municipality(chuva_estacoes, associacao)
df_analitica = base_analitica.build_base(semanal, populacao, area, chuva_municipios, saneamento, ULTIMO_MES)
print(f"{len(df_analitica):,} linhas = {df_analitica['codigo_ibge'].nunique()} municípios × "
      f"{df_analitica['ano_mes'].nunique()} meses ({df_analitica['ano_mes'].min():%m/%Y} a {df_analitica['ano_mes'].max():%m/%Y})")
df_analitica.head()

In [ ]:
df_analitica.to_csv(ARQ_BASE_ANALITICA, index=False)

# Registra quando esta versão da base foi gerada, sem confundir com a data dos dados.
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

caminho_base = Path(ARQ_BASE_ANALITICA)
with caminho_base.open("rb") as arquivo_base:
    checksum_base = hashlib.file_digest(arquivo_base, "sha256").hexdigest()
metadados_base = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "csv_sha256": checksum_base,
}
caminho_base.with_suffix(".metadata.json").write_text(
    json.dumps(metadados_base, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Base analítica final: {len(df_analitica)} linhas, {df_analitica.shape[1]} colunas, "
      f"salva em {ARQ_BASE_ANALITICA}")

## 4. Qualidade dos dados
### 4.1 Lacunas

In [ ]:
lacunas = df_analitica.isna().mean().rename("ausente").to_frame()
lacunas = lacunas[lacunas["ausente"] > 0].sort_values("ausente", ascending=False)
lacunas["ausente"] = lacunas["ausente"].map("{:.1%}".format)
lacunas

**Leitura:**
- **Saneamento:** cobre só os 55 municípios com ETE ativa no Atlas.
- **Temperatura e umidade:** faltam em cerca de 7% das linhas, onde o InfoDengue não tem estação climática associada ao município.
- **Chuva:** falta em cerca de 9%. A maior parte está em 2021–2022 e 2025–2026, quando várias estações do INMET passaram meses sem medir.
- **Nenhuma lacuna é preenchida com zero.** O modelo trata a chuva ausente como ausente, e as linhas sem temperatura ficam sem previsão.

### 4.2 Comparação com os totais estaduais oficiais da DIVE/SC

In [ ]:
nossos = df_analitica.groupby([df_analitica["ano_mes"].dt.year.rename("ano"), df_analitica["ano_mes"].dt.month.rename("mes")])[
    ["casos", "notificacoes"]].sum().reset_index()
# Excluir o mês mais recente do informe antes do cruzamento, sem remover o último mês comum do painel.
referencia_comparavel = referencia.loc[
    referencia["mes"] < referencia.groupby("ano")["mes"].transform("max")
]
comparacao = referencia_comparavel.merge(nossos, on=["ano", "mes"], suffixes=("_dive", "_base"))
anual = comparacao.groupby("ano").agg(meses=("mes", "count"), casos_dive=("casos_provaveis", "sum"), casos_base=("casos", "sum"),
                                      notificacoes_dive=("notificacoes_dive", "sum"), notificacoes_base=("notificacoes_base", "sum"))
anual["dif_casos"] = (anual["casos_base"] / anual["casos_dive"] - 1).map("{:+.1%}".format)
anual["dif_notificacoes"] = (anual["notificacoes_base"] / anual["notificacoes_dive"] - 1).map("{:+.1%}".format)
anual

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
datas = pd.to_datetime(dict(year=comparacao["ano"], month=comparacao["mes"], day=1))
ax.plot(datas, comparacao["casos_provaveis"], marker="o", label="DIVE/SC (total estadual; início dos sintomas)")
ax.plot(datas, comparacao["casos"], marker="o", label="InfoDengue agregado (semana epidemiológica)")
ax.set(ylabel="Casos prováveis por mês", title="Comparação: série InfoDengue × totais oficiais DIVE/SC")
ax.legend()
plt.show()

**Leitura:**
- A DIVE/SC é a referência oficial para os totais estaduais publicados; o InfoDengue fornece a série municipal usada no painel e no modelo. Esta comparação não torna as duas fontes equivalentes nem ajusta uma à outra.
- As diferenças refletem fontes e datas de corte distintas, semana epidemiológica no InfoDengue versus mês de início dos sintomas na DIVE e revisões retrospectivas/classificações posteriores.
- Para evitar comparar um mês possivelmente parcial, exclui-se o último mês de cada informe **antes** do cruzamento. Em 2026, exclui novembro do informe 11/2026 e compara janeiro–outubro, período disponível também no painel.
- Nos períodos comparáveis atuais, os casos prováveis do painel ficam 7,85% acima da DIVE em 2024, 12,60% em 2025 e 18,26% em 2026; as notificações ficam 4,29%, 6,00% e 5,44% acima, respectivamente.

## 5. Análise exploratória: hipóteses

Cada hipótese é examinada com gráficos e com a correlação de Spearman, que mede associação
monotônica e é robusta a valores extremos. As associações não provam causa. A dengue depende de
fatores não medidos aqui, como a circulação de sorotipos e a imunidade da população.

In [ ]:
from scipy.stats import spearmanr

df = df_analitica.sort_values(["codigo_ibge", "ano_mes"]).copy()
df["mes"] = df["ano_mes"].dt.month
df["epidemico"] = df["incidencia_100k"] >= 300
grupo = df.groupby("codigo_ibge")
df["incidencia_lag1"] = grupo["incidencia_100k"].shift(1)
df["precipitacao_lag1"] = grupo["precipitacao_total"].shift(1)
df["temperatura_lag1"] = grupo["temperatura_media"].shift(1)
ativos = df[df["ano"] >= 2022]  # anos com circulação relevante do vírus em SC


def associacao(x, y, dados):
    d = dados[[x, y]].dropna()
    rho, p = spearmanr(d[x], d[y])
    return f"Spearman ρ = {rho:.2f} (p = {p:.1g}, n = {len(d):,})"


def media_por_faixa(dados, coluna, faixas, rotulo):
    tabela = dados.assign(faixa=pd.cut(dados[coluna], faixas)).groupby("faixa", observed=True).agg(
        incidencia_media=("incidencia_100k", "mean"), meses_epidemicos=("epidemico", "mean"), registros=("incidencia_100k", "size"))
    tabela.index = tabela.index.astype(str)
    fig, ax = plt.subplots()
    ax.bar(tabela.index, tabela["incidencia_media"], color="#e8590c")
    ax.set(xlabel=rotulo, ylabel="Casos por 100 mil (média)")
    plt.xticks(rotation=30)
    plt.show()
    return tabela.assign(meses_epidemicos=tabela["meses_epidemicos"].map("{:.1%}".format)).round(1)

### H1 — Sazonalidade: a dengue concentra-se em alguns meses do ano

In [ ]:
matriz = df.pivot_table(index="ano", columns="mes", values="casos", aggfunc="sum")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
media_mes = df.groupby(["ano", "mes"])["casos"].sum().groupby("mes").mean()
axes[0].bar(media_mes.index, media_mes.values, color="#e8590c")
axes[0].set(xlabel="Mês", ylabel="Casos prováveis (média anual)", xticks=range(1, 13), title="Média por mês do ano")
imagem = axes[1].imshow(np.log10(matriz + 1), aspect="auto", cmap="YlOrRd")
axes[1].set(xticks=range(12), xticklabels=range(1, 13), yticks=range(len(matriz)), yticklabels=matriz.index,
            xlabel="Mês", title="Casos prováveis por mês e ano (escala log)")
fig.colorbar(imagem, ax=axes[1], label="log10(casos + 1)")
plt.tight_layout(); plt.show()
participacao = (media_mes.nlargest(3).sum() / media_mes.sum())
print(f"Três meses com mais casos: {sorted(media_mes.nlargest(3).index.tolist())} — {participacao:.0%} dos casos do ano")

**Confirmada.** Em média, **81% dos casos do ano acontecem entre março e maio**, no fim do verão e
no outono. O quadro ano × mês também mostra a mudança de patamar a partir de 2022: as epidemias
de 2022, 2023 e principalmente 2024 superam em uma ou duas ordens de grandeza os anos anteriores.
Por isso o mês do calendário entra no modelo, e a análise das demais hipóteses usa 2022 em diante.

### H2 — Temperatura: meses mais quentes têm maior incidência (inclusive com defasagem de um mês)

In [ ]:
print("Temperatura do mês:", associacao("temperatura_media", "incidencia_100k", ativos))
print("Temperatura do mês anterior:", associacao("temperatura_lag1", "incidencia_100k", ativos))
media_por_faixa(ativos, "temperatura_lag1", [10, 14, 16, 18, 20, 22, 24, 26, 30], "Temperatura média do mês anterior (°C)")

**Confirmada, e mais forte com defasagem.** A temperatura do mês anterior tem associação maior
(ρ = 0,25) que a do próprio mês (ρ = 0,12). Isso é coerente com o ciclo do mosquito e do vírus,
que leva semanas. A incidência média sobe de menos de 1 caso por 100 mil, abaixo de 14 °C, para
177 casos por 100 mil entre 24 °C e 26 °C, e a parcela de meses epidêmicos vai de 0% a 10%. A
queda acima de 26 °C reúne poucos registros. As duas temperaturas entram no modelo.

### H3 — Chuva: a chuva do mês anterior acompanha mais casos no mês seguinte

In [ ]:
print("Chuva do mês:", associacao("precipitacao_total", "incidencia_100k", ativos))
print("Chuva do mês anterior:", associacao("precipitacao_lag1", "incidencia_100k", ativos))
media_por_faixa(ativos, "precipitacao_lag1", [0, 50, 100, 150, 200, 300, 1200], "Chuva no mês anterior (mm)")

**Não confirmada como relação simples.** A correlação da chuva com a incidência é praticamente
nula (ρ = 0,02), tanto no mês quanto com defasagem. As faixas não mostram tendência clara: meses
muito secos e muito chuvosos têm médias parecidas. A chuva pode criar criadouros, mas também os
lava, e o efeito depende do armazenamento de água em cada município. Ela permanece no modelo,
que pode captar interações, mas com peso pequeno.

### H4 — Umidade: umidade mais alta acompanha maior incidência

In [ ]:
print(associacao("umidade_media", "incidencia_100k", ativos))
media_por_faixa(ativos, "umidade_media", [50, 70, 75, 80, 85, 90, 100], "Umidade média do ar (%)")

**Confirmada de forma fraca.** A associação é positiva (ρ = 0,15), e meses com umidade acima de
75% têm incidência média de 3 a 5 vezes a dos meses mais secos. Parte desse efeito se confunde
com a estação do ano, que é quente e úmida ao mesmo tempo.

### H5 — Densidade demográfica: municípios mais densos têm maior incidência

In [ ]:
por_municipio = ativos.groupby("codigo_ibge").agg(
    densidade=("densidade_demografica", "mean"), casos=("casos", "sum"), populacao=("populacao", "mean"),
    meses_epidemicos=("epidemico", "sum"))
por_municipio["incidencia_acumulada"] = por_municipio["casos"] * 1e5 / por_municipio["populacao"]
print(associacao("densidade", "incidencia_acumulada", por_municipio))
fig, ax = plt.subplots()
ax.scatter(por_municipio["densidade"], por_municipio["incidencia_acumulada"] + 1, s=12, alpha=0.6, color="#e8590c")
ax.set(xscale="log", yscale="log", xlabel="Habitantes por km² (log)",
       ylabel="Casos por 100 mil, 2022 em diante (log)", title="Um ponto por município")
plt.show()

**Confirmada.** Municípios mais densos acumularam mais casos por 100 mil habitantes desde 2022
(ρ = 0,40 entre os 295 municípios). Isso é coerente com a maior proximidade entre pessoas e a
maior oferta de criadouros urbanos. A densidade entra no modelo junto com o tamanho da população.

### H6 — Saneamento: melhor tratamento de esgoto acompanha menor incidência

In [ ]:
com_ete = por_municipio.join(saneamento.set_index("codigo_ibge")).dropna(subset=["remocao_dbo_media"])
print(f"{len(com_ete)} municípios com ETE ativa no Atlas (cobertura parcial)")
print("Remoção de DBO × incidência:", associacao("remocao_dbo_media", "incidencia_acumulada", com_ete))
print("Densidade × incidência (mesmos municípios):", associacao("densidade", "incidencia_acumulada", com_ete))
fig, ax = plt.subplots()
ax.scatter(com_ete["remocao_dbo_media"] * 100, com_ete["incidencia_acumulada"], s=18, color="#1c7ed6")
ax.set(xlabel="Remoção média de DBO nas ETEs (%)", ylabel="Casos por 100 mil, 2022 em diante")
plt.show()

**Não confirmada com os dados disponíveis.** Entre os 55 municípios com ETE ativa no Atlas, a
remoção de DBO não se associa à incidência (ρ = 0,09, p = 0,5), enquanto a densidade continua
associada nesse mesmo grupo (ρ = 0,50). O Atlas cobre poucos municípios e mede a qualidade do
tratamento, não o acesso da população ao esgoto nem à água encanada. Por isso o saneamento não
entra no modelo.

### H7 — Dependência temporal: os casos de um mês antecipam os do mês seguinte

In [ ]:
print(associacao("incidencia_lag1", "incidencia_100k", df))
transicao = df.dropna(subset=["incidencia_lag1"]).assign(
    anterior=lambda d: pd.cut(d["incidencia_lag1"], [-1, 0, 10, 50, 100, 300, np.inf],
                              labels=["0", "até 10", "10–50", "50–100", "100–300", "300 ou mais"]))
tabela = transicao.groupby("anterior", observed=True).agg(
    registros=("epidemico", "size"), chance_mes_epidemico=("epidemico", "mean"))
tabela["chance_mes_epidemico"] = tabela["chance_mes_epidemico"].map("{:.1%}".format)
tabela.index.name = "casos por 100 mil no mês anterior"
tabela

**Confirmada — é o fator mais forte.** A incidência de um mês tem correlação de 0,60 com a do mês
anterior. Quando o mês anterior teve 300 casos ou mais por 100 mil habitantes, **62% dos meses
seguintes também foram epidêmicos**; quando não houve casos, apenas 0,1%. A dengue se espalha a
partir de quem já está doente. Por isso o modelo usa os casos dos meses anteriores no próprio
município, nos municípios vizinhos e no estado.

## 6. Conclusões


| Hipótese | Resultado | Uso no modelo |
|---|---|---|
| H1 Sazonalidade | Confirmada: 81% dos casos de março a maio | Mês do ano |
| H2 Temperatura | Confirmada, mais forte com um mês de defasagem | Temperatura do mês e do mês anterior |
| H3 Chuva | Sem relação simples | Chuva do mês e dos dois anteriores, peso pequeno |
| H4 Umidade | Associação fraca | Umidade do mês e do mês anterior |
| H5 Densidade | Confirmada | Densidade e tamanho da população |
| H6 Saneamento | Não confirmada (55 municípios) | Não entra |
| H7 Dependência temporal | Confirmada, o fator mais forte | Casos dos meses anteriores no município, nos vizinhos e no estado |

A base analítica gerada aqui é a entrada do notebook de modelagem (`notebook_modelo_baseline.ipynb`).